# 🤖 Building a Local AI ChatBot with Ollama & Gradio

Welcome to this hands-on tutorial! In this notebook, you will build an interactive AI ChatBot that runs **100% locally on your machine**—private, fast, and with zero API costs.

### 🛠️ Architecture
- **[Ollama](https://ollama.com/)**: Serves open-weight LLMs (like Llama 3.2, DeepSeek, Mistral) on your machine.
- **[Gradio](https://gradio.app/)**: Creates a sleek, responsive web UI with built-in streaming chat.
- **Python `ollama` SDK**: Connects the frontend UI to the local model server.


## 📦 Step 1: Install Dependencies

Run the cell below to install `ollama`, `gradio`, `openai`, and `python-dotenv` in your environment.


In [ ]:
%pip install -q ollama gradio python-dotenv openai


## 📥 Step 2: Download (Pull) an Ollama Model

Before running the code, ensure the Ollama service is active. If you haven't already downloaded a model, run this command in your terminal:

```bash
ollama pull llama3.2
```

Here is what the model download progress looks like in your terminal:

![Ollama Pull Terminal](assets/ollama_pull_terminal.jpg)

## 🔍 Step 3: Verify Connection & List Local Models

Let's verify that Python can connect to the local Ollama instance and view all available models.

In [49]:
import ollama

# Query available models from the local Ollama daemon
try:
    response = ollama.list()
    models = response.get("models", [])
    print(f"✅ Successfully connected to Ollama! Found {len(models)} local model(s):\n")
    for m in models:
        name = m.get("model") or m.get("name")
        size_gb = m.get("size", 0) / (1024 ** 3)
        print(f" • {name} ({size_gb:.2f} GB)")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("Ensure Ollama is running: run `ollama serve` in a terminal.")

✅ Successfully connected to Ollama! Found 6 local model(s):

 • gpt-oss:20b (12.85 GB)
 • llama3.2:latest (1.88 GB)
 • deepseek-r1:1.5b (1.04 GB)
 • gpt-oss:latest (12.85 GB)
 • phi3:latest (2.03 GB)
 • gemma3:270m (0.27 GB)


## 💬 Step 4: Basic Single-Turn Prompt

Let's send a simple prompt to `llama3.2` and inspect the structured response.

In [ ]:
prompt = "Explain what a Large Language Model is in one simple sentence."

response = ollama.chat(
    model="llama3.2",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("🤖 Response:")
print(response["message"]["content"])

## ⚡ Step 5: Streaming Token Responses in Real-Time

For chatbots, streaming response tokens as they are generated provides immediate feedback and feels much faster than waiting for completion.

In [ ]:
stream = ollama.chat(
    model="llama3.2",
    messages=[
        {"role": "user", "content": "Write a 4-line poem about programming with Python."}
    ],
    stream=True
)

print("Streaming: ", end="", flush=True)
for chunk in stream:
    token = chunk.get("message", {}).get("content") or ""
    if token:
        print(token, end="", flush=True)
print()


## 🔄 Step 5b: Using the `OpenAI` Python SDK with Ollama

Ollama exposes a fully OpenAI-compatible API endpoint at `http://localhost:11434/v1`.
This means you can use the standard `from openai import OpenAI` client with local models without changing your application code!

> **Note:** This is **not** ChatGPT. The `openai` package is only the client library; `base_url` sends the requests to your local Ollama server, which runs the local model (`llama3.2`).

In [ ]:
from openai import OpenAI

# Point OpenAI client to your local Ollama server
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # Required by SDK, but ignored by local Ollama
)

streamResponse = client.chat.completions.create(
    model="llama3.2",  # local Ollama model (not ChatGPT)
    messages=[
        {"role": "user", "content": "What is the capital of France? please describe about the city in not more than 4 lines."}
    ],
    stream=True
)

for chunk in streamResponse:
    token = chunk.choices[0].delta.content or ""
    if token:
        print(token, end="", flush=True)
print()


The capital of France is Paris. Paris, the City of Light, is known for its stunning architecture, world-class museums, and romantic atmosphere. The city is home to iconic landmarks like the Eiffel Tower and Notre-Dame Cathedral, attracting millions of tourists each year. Paris is also famous for its fashion, art, and culinary scene.None

## ☁️ Step 5c: Using the Official OpenAI ChatGPT API

Now let's compare local Ollama with the official cloud-hosted **OpenAI ChatGPT API**.

Because Ollama adopts OpenAI's API specification, switching between local models and ChatGPT in Python requires only two small changes:
1. **Remove `base_url`** (it automatically defaults to `https://api.openai.com/v1`).
2. **Provide your real `OPENAI_API_KEY`** (loaded securely from your `.env` file with `python-dotenv`).

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load API key from local .env file
load_dotenv()
# The OPENAI_API_KEY that should be saved in your .venv file please see the instruction for creating the  
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("⚠️ OPENAI_API_KEY not found in .env.")
    print("To test the official OpenAI ChatGPT API:")
    print("1. Create a .env file from .env.example:  cp .env.example .env")
    print("2. Add your secret key:  OPENAI_API_KEY=sk-...")
else:
    # Initialize standard client connecting to OpenAI cloud
    client = OpenAI(api_key=api_key)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful and concise AI assistant."},
            {"role": "user", "content": "Explain what an API is in one short sentence."}
        ]
    )

    print("🤖 ChatGPT (gpt-4o-mini) Response:")
    print(response.choices[0].message.content)


## 🎨 Step 6: Build the Gradio Web ChatBot Interface

Now we assemble everything into an interactive web application using [Gradio](https://www.gradio.app/). For comprehensive documentation, component guides, and advanced examples, visit the official [Gradio website](https://www.gradio.app/).

> **💡 Note on Alternatives:** Another widely used Python framework for building machine learning web apps is [Streamlit](https://streamlit.io/). While Streamlit is also a great option, we will use **Gradio** in this tutorial because of its native `gr.ChatInterface`, effortless real-time token streaming, and built-in chat history handling.

### Key Implementation Details:
1. **System Prompt**: Gives the assistant a consistent persona and instructions.
2. **Conversation Context**: Carries previous user and assistant turns so follow-up questions work naturally.
3. **Streaming Generator**: Yields updated responses chunk-by-chunk directly into the chat bubble.

Here is what the resulting Gradio ChatBot looks like:

![Gradio ChatBot Screenshot](assets/gradio_chatbot_ui.jpg)

In [ ]:
import os
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI

# Load environment variables and configure OpenAI client for local Ollama endpoint
load_dotenv()
OLLAMA_HOST = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

client = OpenAI(
    base_url=f"{OLLAMA_HOST}/v1",
    api_key="ollama"  # Required by OpenAI SDK, but ignored by local Ollama
)

def to_text(content) -> str:
    """Newer Gradio versions store message content as a list of parts
    (e.g. [{'text': '...', 'type': 'text'}]); join them into plain text."""
    if isinstance(content, list):
        return "".join(p.get("text", "") for p in content if isinstance(p, dict))
    return str(content)

def chat_with_ollama(message: str, history: list):
    """Processes user messages, maintains history, and streams response chunks via client.chat.completions."""
    messages = [
        {"role": "system", "content": "You are a helpful, intelligent, and friendly AI assistant."}
    ]

    print(f"message is : {message}")
    print(f"History is : {history}")
    # Reconstruct multi-turn chat history (supports both dict and tuple formats)
    # The model is stateless, so we replay earlier turns on every call
    for turn in history:
        if isinstance(turn, dict):
            messages.append(
                {"role": turn.get("role", "user"), "content": to_text(turn.get("content", ""))}
            )
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            messages.append({"role": "user", "content": to_text(turn[0])})
            messages.append({"role": "assistant", "content": to_text(turn[1])})

    # Append new user prompt
    messages.append({"role": "user", "content": message})

    # Stream response using client.chat.completions.create
    accumulated_response = ""
    try:
        stream = client.chat.completions.create(
            model="llama3.2",  # local Ollama model (not ChatGPT)
            messages=messages,
            stream=True
        )
        for chunk in stream:
            # Safe extraction: ignore chunks where delta.content is None (e.g. final stop chunk)
            if chunk.choices and chunk.choices[0].delta.content:
                accumulated_response += chunk.choices[0].delta.content
                yield accumulated_response
    except Exception as err:
        yield f"⚠️ Error contacting Ollama: {err}\nPlease ensure Ollama is running (`ollama serve`)."

# Construct the Gradio chat interface
demo = gr.ChatInterface(
    fn=chat_with_ollama,
    title="🦙 Local Ollama AI ChatBot (OpenAI SDK)",
    examples=[
        "Explain what a neural network is in simple terms.",
        "Write a Python function to find prime numbers up to n.",
        "Give me 3 practical project ideas combining LLMs and databases.",
    ],
    flagging_mode="never"
)

# Launch the UI 
demo.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


message is : Explain what a neural network is in simple terms.
History is : []
message is : Explain what a neural network is in simple terms.
History is : []
message is : Hello my name is Mukesh
History is : [{'role': 'user', 'metadata': None, 'content': [{'text': 'Explain what a neural network is in simple terms.', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': 'A neural network is a computer system that’s designed to *learn* from examples, just like how we humans learn from experience. Think of it like a very simplified brain made of tiny “neurons” (little decision points) that are all connected.  \n\n**Here’s the basic idea in plain English:**\n\n1. **Neurons:** Each neuron receives some inputs, does a simple calculation, and sends an output to the next layer of neurons.  \n2. **Layers:** Many of these neurons are grouped into layers. The first layer takes the raw input (like pixels of a picture), the middle layers process that infor

## 🚀 Next Steps & Enhancements

- **Change Models**: Try changing `model="llama3.2"` to `"deepseek-r1:1.5b"` for reasoning or `"qwen2.5-coder:7b"` for coding.
- **Public Shareable Link**: Add `demo.launch(share=True)` to create a temporary public 72-hour URL to share with others.
- **Custom System Persona**: Change the system message to fit your use case (e.g., Code Reviewer, Tutor, or Data Analyst).
- **Calling the External API**: connect to the remote API from the Chat GPT, Anthropic, Google, grok etc. by confuguring your API key
- **Fun Questions**: Given the LLM are stateless in conversation, given the above snippet also keep the history trying telling about your name and what you do and ask the questions later.. see the that it is ables to reconize it from the previous histoy as we are passing the previous one as part of the context.
